# Getting Started

This guide walks through a complete first workflow with `XAIMetrics.jl`:

1. define a model and explanation method,
2. prepare batched input,
3. evaluate robustness metrics,
4. customize perturbation and normalization behavior.

> **Note**
>
> `XAIMetrics.jl` evaluates explanations generated by `ExplainableAI.jl` methods.

## Prerequisites

In [1]:
using XAIMetrics
using ExplainableAI
using Flux
using Random

Create a simple classifier and an explanation method:

In [2]:
model = Chain(
    Flux.flatten,
    Dense(28 * 28, 64, relu),
    Dense(64, 10),
)

method = InputTimesGradient(model)

ExplainableAI.InputTimesGradient{Flux.Chain{Tuple{typeof(Flux.flatten), Flux.Dense{typeof(NNlib.relu), Matrix{Float32}, Vector{Float32}}, Flux.Dense{typeof(identity), Matrix{Float32}, Vector{Float32}}}}, ADTypes.AutoZygote}(Chain(flatten, Dense(784 => 64, relu), Dense(64 => 10)), ADTypes.AutoZygote())

## Prepare input and targets
`evaluate` expects batched input with batch dimension last.
For image data in Flux, this is typically `WHCN`.

In [3]:
batch_size = 8
x = rand(Float32, 28, 28, 1, batch_size)
y = rand(1:10, batch_size)

8-element Vector{Int64}:
 7
 7
 7
 7
 4
 6
 5
 3

> **Input shape**
>
> The last dimension is treated as batch dimension by `XAIMetrics.jl`.

## First metric evaluation
Start with `AvgSensitivity`:

In [4]:
metric = AvgSensitivity(
    nr_samples = 30,
    perturb_config = PerturbationConfig(
        uniform_noise!;
        params = (; lower = 0.05, upper = 0.1),
    ),
)

scores = evaluate(metric, method, x; y = y)

8-element Vector{Float32}:
 0.06908145
 0.37426534
 0.086791426
 0.07030134
 0.2311088
 0.24885274
 0.18895096
 0.15331751

`scores` is a vector with one score per sample in the batch.

Then continue with `LocalLipschitzEstimate`:

In [5]:
lle_metric = LocalLipschitzEstimate(
    nr_samples = 50,
    perturb_config = PerturbationConfig(
        gaussian_perturbation!;
        params = (; std = 0.2),
    ),
    return_nan_when_prediction_changes = false,
)

lle_scores = evaluate(lle_metric, method, x; y = y)

8-element Vector{Float32}:
 0.61613643
 0.6600307
 0.69747406
 0.65111417
 0.6310722
 0.7701345
 0.68138504
 0.7329931

## Configuring behavior
### Perturbations

In [6]:
cfg_uniform = PerturbationConfig(
    uniform_noise!;
    params = (; lower = 0.03, upper = 0.08))

PerturbationConfig{typeof(uniform_noise!), Distributions.Uniform{Float64}}(XAIMetrics.uniform_noise!, Distributions.Uniform{Float64}(a=0.03, b=0.08))

### Normalization

In [7]:
norm_cfg = NormalizationConfig(
    normalize = true,
    normalize_func = normalize_by_max_abs,
)

metric_norm = AvgSensitivity(
    nr_samples = 30,
    perturb_config = cfg_uniform,
    normalize_config = norm_cfg,
)

AvgSensitivity{typeof(difference), typeof(XAIMetrics.columnwise_l2_norm), typeof(XAIMetrics.columnwise_l2_norm)}(30, XAIMetrics.difference, XAIMetrics.columnwise_l2_norm, XAIMetrics.columnwise_l2_norm, false, PerturbationConfig{typeof(uniform_noise!), Distributions.Uniform{Float64}}(XAIMetrics.uniform_noise!, Distributions.Uniform{Float64}(a=0.03, b=0.08)), NormalizationConfig{typeof(normalize_by_max_abs)}(true, XAIMetrics.normalize_by_max_abs))

## Inspecting perturbations
You can also apply perturbations directly to a small input and print the result.

In [8]:
Random.seed!(7)

x_demo = Float32[0.2, -0.4, 0.8]
x_demo_pert = similar(x_demo)

demo_pert_cfg = PerturbationConfig(uniform_noise!; params = (; lower = 0.03, upper = 0.08))
perturb_input!(x_demo_pert, x_demo, demo_pert_cfg)

println("x_demo          = ", x_demo)
println("x_demo_pert     = ", x_demo_pert)
println("perturbation Δ  = ", x_demo_pert .- x_demo)

x_demo          = Float32[0.2, -0.4, 0.8]
x_demo_pert     = Float32[0.24481063, -0.35781026, 0.87926626]
perturbation Δ  = Float32[0.044810623, 0.042189747, 0.07926625]


## Inspecting normalization
Normalization can be inspected the same way.

In [9]:
a_demo = Float32[-2.0, 0.0, 1.0, 4.0]
a_norm = normalize_by_max_abs(a_demo)

println("a_demo                  = ", a_demo)
println("normalize_by_max_abs    = ", a_norm)
println("maximum(abs, a_demo)    = ", maximum(abs, a_demo))
println("maximum(abs, a_norm)    = ", maximum(abs, a_norm))

a_demo                  = Float32[-2.0, 0.0, 1.0, 4.0]
normalize_by_max_abs    = Float32[-0.5, 0.0, 0.25, 1.0]
maximum(abs, a_demo)    = 4.0
maximum(abs, a_norm)    = 1.0


## Handling prediction changes
Both robustness metrics support `return_nan_when_prediction_changes`:

- `true`: keep changed-prediction cases as `NaN`
- `false`: aggregate over available finite scores

This is useful when you want to separate explanation instability from class-change effects.

## Next steps

- Robustness metric reference: [Robustness](../metrics/robustness.md)
- Configuration overview: [Configurations](../configurations.md)
- Function-level APIs: [Normalizations](../normalizations.md), [Perturbations](../perturbations.md), [Similarities](../similarities.md)

---

*This notebook was generated using [Literate.jl](https://github.com/fredrikekre/Literate.jl).*